# Codification 1 : machine learning de septembre, montants au logarithme

**Variante de [`01_ml_septembre.ipynb`](01_ml_septembre.ipynb), un seul changement** : les montants dus (`BILL_AMT1` à `BILL_AMT6`), payés (`PAY_AMT1` à `PAY_AMT6`) et le plafond (`LIMIT_BAL`) passent au **logarithme**, puis sont standardisés pour la régression logistique et le MLP. Les soldes pouvant être négatifs (avoirs), le logarithme est **signé** : signe(x) × log(1 + |x|), qui garde 0 à 0 et le signe des avoirs. Question : les montants très étirés perturbent-ils moins les modèles une fois au logarithme ? Le reste (données, découpage, modèles, paramètres, tableaux) est identique au notebook 01.

Première étape de l'essai de correction de la codification 1 par machine learning (journal : [`tableau_resultats.md`](tableau_resultats.md)). Essai **bonus et indépendant** des autres branches de machine learning du projet : il n'en reprend aucun choix (variables, encodages, transformations, réglages).

**Hypothèse testée, une seule** : la codification 1 est un **code d'attente**. La banque la pose à la place d'une vraie codification (-2, -1, 0, ou un retard de 2 et plus), qu'elle n'a pas encore tranchée. Si un modèle sait dire ce que cache un 1, l'hypothèse tient ; s'il ne sait pas décider, c'est que le 1 est autre chose.

**Données** : niveau 0 du nettoyage (`data/creditcard_pret_ingestion.csv`, 30 000 clients), tel quel ; **`dpnm` est retiré dès le chargement** : il n'est utilisé à aucun moment. Les corrections manuelles des codifications 1 d'avril à août ([`02_corrections_manuelles.ipynb`](02_corrections_manuelles.ipynb)) seront appliquées **après** le machine learning, en correctif, s'il est concluant. Les codifications corrigées serviront de variables à un futur modèle de défaut ; reconstruites avec `dpnm`, elles contiendraient la cible.

**Critère fixé avant le lancement** :
- **A, le modèle sait-il lire une codification ?** Sur le test, il doit faire nettement mieux que la règle naïve « même codification qu'en août » (`PAY_1` = `PAY_2`). Sinon, on s'arrête : rien ne peut être conclu sur les 1.
- **B, un 1 ressemble-t-il à une codification ordinaire ?** Sur les `PAY_1` = 1, le modèle doit trancher aussi nettement que sur le test. S'il hésite beaucoup plus, l'hypothèse tombe : le 1 est autre chose.

## 1. Données

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
PAY = [f"PAY_{n}" for n in range(1, 7)]
print(f"Clients du niveau 0 : {len(df)} ; colonnes : {df.shape[1]} (dpnm retirée) ; PAY_1 = 1 : {int((df['PAY_1'] == 1).sum())}")

# Codifications 1 d'avril à août : leurs clients ont tous PAY_1 = 1, ils restent donc hors du train et du test
anciens_1 = (df[PAY[1:]] == 1).any(axis=1)
assert (df.loc[anciens_1, "PAY_1"] == 1).all()
print(f"Clients avec une codification 1 d'avril à août : {int(anciens_1.sum())}, tous codifiés 1 aussi en septembre")

Clients du niveau 0 : 30000 ; colonnes : 24 (dpnm retirée) ; PAY_1 = 1 : 3688
Clients avec une codification 1 d'avril à août : 28, tous codifiés 1 aussi en septembre


## 2. Septembre : ce que la banque voyait, et ce qu'on cherche à retrouver

**Cible** : la codification de `PAY_1`, **telle quelle**, de -2 à 8 (une classe par codification, sans regrouper les retards) : le niveau du retard fait partie de ce que le modèle doit comprendre. Un essai avec les retards regroupés viendra après, seulement si le résultat est peu probant.

**Variables** : les colonnes du dataset d'origine (niveau 0), sauf `ID` (un identifiant), `PAY_1` (la cible), `dpnm` et les données personnelles ; **aucune colonne ajoutée** :
- les codifications d'avril à août (`PAY_2` à `PAY_6`), telles quelles : les 28 clients qui y ont une codification 1 l'ont tous aussi en septembre, ils ne sont donc ni dans le train ni dans le test ;
- les soldes de fin de mois (`BILL_AMT1` à `BILL_AMT6`), les paiements (`PAY_AMT1` à `PAY_AMT6`) et le plafond (`LIMIT_BAL`).

Les données personnelles (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`) sont retirées : elles ne sont normalement pas pertinentes pour une codification, qui est un statut de paiement.

Ce ML est un **essai indépendant** : il ne reprend aucun choix des autres branches de ML du projet.

**Exemples** : les clients dont `PAY_1` n'est pas un 1, coupés en 80/20 (train / test, `stratify` sur la classe, `random_state=42`). Les `PAY_1` = 1 sont mis de côté : ce sont eux qu'on cherche à trancher.

In [3]:
from sklearn.model_selection import train_test_split

# Variables : colonnes du dataset d'origine (niveau 0), sauf ID (identifiant), PAY_1 (la cible), dpnm et les données personnelles ; aucune colonne ajoutée
PERSONNELLES = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]   # retirées : non pertinentes pour une codification de paiement
NUMERIQUES = ["LIMIT_BAL"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
CATEGORIELLES = PAY[1:]
VARIABLES = CATEGORIELLES + NUMERIQUES
assert sorted(VARIABLES + ["ID", "PAY_1"] + PERSONNELLES) == sorted(df.columns)
mask_1 = df["PAY_1"] == 1
connus = df[~mask_1]
a_trancher = df[mask_1]

X_train, X_test, y_train, y_test = train_test_split(
    connus[VARIABLES], connus["PAY_1"], test_size=0.2, stratify=connus["PAY_1"], random_state=42)
CLASSES = sorted(connus["PAY_1"].unique())   # toutes les codifications de PAY_1, sauf 1

repartition = pd.DataFrame({
    "Train": y_train.value_counts(),
    "Test": y_test.value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)
repartition.loc["Total"] = repartition.sum()
print(f"PAY_1 connus : {len(connus)} (train {len(X_train)}, test {len(X_test)}) ; PAY_1 = 1 à trancher : {len(a_trancher)}")
print(f"Variables ({len(VARIABLES)}) : {VARIABLES}")
display(repartition)

PAY_1 connus : 26312 (train 21049, test 5263) ; PAY_1 = 1 à trancher : 3688
Variables (18) : ['PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'LIMIT_BAL', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


,Train,Test
PAY_1,,
-2,2207,552
-1,4549,1137
0,11789,2948
2,2133,534
3,258,64
4,61,15
5,21,5
6,9,2
7,7,2


**Comment lire** : nombre de clients par codification de `PAY_1`, dans le train et dans le test. Les proportions sont les mêmes dans les deux (`stratify`). Le test n'est plus touché jusqu'au critère A.

## 3. Les niveaux de retard d'août à septembre (train)

`PAY_2` compte des retards importants (3 à 8) qui repassent à 2 en septembre, ce qui peut perturber les modèles. Le tableau croise, sur les clients du train, la codification d'août (en ligne) et celle de septembre (en colonne) : les codifications saines (-2, -1, 0) sont réunies, les niveaux de retard sont détaillés.

In [4]:
def libelle_retard(codifications):
    # Niveaux de retard détaillés, codifications saines réunies
    return codifications.astype(int).astype(str).where(codifications >= 2, "sain (-2, -1, 0)")

croisement = pd.crosstab(libelle_retard(X_train["PAY_2"]).rename("Août (PAY_2)"),
                         libelle_retard(y_train).rename("Septembre (PAY_1)"), margins=True, margins_name="Total")
display(croisement)

Septembre (PAY_1),2,3,4,5,6,7,8,"sain (-2, -1, 0)",Total
Août (PAY_2),,,,,,,,,
2,1266,216,0,0,0,0,0,326,1808
3,57,35,46,0,0,0,0,35,173
4,11,6,12,20,0,0,0,4,53
5,0,1,3,0,9,0,0,1,14
6,0,0,0,1,0,7,0,0,8
7,0,0,0,0,0,0,15,0,15
"sain (-2, -1, 0)",799,0,0,0,0,0,0,18179,18978
Total,2133,258,61,21,9,7,15,18545,21049


**Comment lire** : chaque case donne le nombre de clients du train passés de la codification d'août (ligne) à celle de septembre (colonne). La diagonale correspond aux retards gardés au même niveau ; sous la diagonale, à gauche, les retards qui baissent (par exemple de 3 en août à 2 en septembre) ; au-dessus, ceux qui montent. La ligne et la colonne « Total » donnent le nombre de codifications de chaque niveau, en août et en septembre.

## 4. Premier passage : cinq modèles aux paramètres standards (train / validation)

Le train est coupé une seconde fois, en 80/20 : le modèle **apprend** sur la première partie (train) et il est **noté** sur la seconde (validation), qu'il n'a jamais vue. Le test reste de côté.

Cinq modèles, avec leurs paramètres standards :
- **Régression logistique** et **MLP** : les codifications en une colonne par valeur ; les montants dus, payés et le plafond au logarithme signé, puis mis à l'échelle standard ;
- **Arbre de décision** (profondeur limitée à 5) : la codification est produite par le système de la banque, très probablement selon une règle de gestion sur les montants et l'historique ; une règle de gestion **est** un arbre de décision. Si un arbre peu profond, que l'on peut lire, fait presque aussi bien que les autres modèles, la banque code avec une règle simple ; catégories en une colonne par valeur, montants et plafond au logarithme signé ;
- **RandomForest** : catégories en une colonne par valeur, montants et plafond au logarithme signé ;
- **CatBoost** : catégories déclarées comme telles (`cat_features`, traitement natif de CatBoost), montants et plafond au logarithme signé.

Les arbres coupent à des seuils : un logarithme ne change pas l'ordre des montants, donc presque pas leurs coupes. Leurs résultats doivent rester proches de ceux du notebook 01 ; un écart net signalerait autre chose que l'effet du logarithme.

Les codifications `PAY_2` à `PAY_6` sont des **catégories pour les cinq modèles** : l'ordre entre -2, -1 et 0 n'est pas documenté, un modèle ne doit pas le supposer. Paramètres des modèles : ceux par défaut de chaque bibliothèque (seuls `max_iter`, pour que la régression logistique et le MLP aillent au bout, et `random_state`, pour des résultats reproductibles, sont fixés).

Le modèle apprend les codifications détaillées (-2 à 8). Une prédiction est comptée **bonne** si elle tombe dans la bonne famille : **-2**, **-1**, **0** ou **2 et plus** (retard, tous niveaux réunis).

Deux points de comparaison, sans apprentissage :
- **« même codification qu'en août »** (`PAY_1` = `PAY_2`) : la règle naïve à battre ;
- **« toujours 0 »** : la codification la plus fréquente. Le pourcentage global qu'elle obtient, sans rien comprendre, est le plancher.

In [5]:
import time
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FAMILLES = [-2, -1, 0, 2]
NOMS_FAMILLES = {-2: "-2", -1: "-1", 0: "0", 2: "2 et plus"}

def famille_pay(codifications):
    # Codification -> famille : -2, -1, 0 gardés, 2 et plus réunis en 2
    v = np.asarray(codifications).ravel().astype(int)
    return np.where(v >= 2, 2, v)

def bonnes_predictions(vrai, predit):
    # Pourcentage de bonnes prédictions (bonne famille), en tout et dans chaque famille
    v, p = famille_pay(vrai), famille_pay(predit)
    resultat = {"global": (v == p).mean() * 100}
    for f in FAMILLES:
        resultat[f] = (p[v == f] == f).mean() * 100
    return resultat

MONTANTS_DUS_PAYES = [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)] + ["LIMIT_BAL"]

def log_montants(X):
    # Logarithme signé des montants dus, payés et du plafond : signe(x) × log(1 + |x|) ; les codifications ne changent pas
    X = X.copy()
    X[MONTANTS_DUS_PAYES] = np.sign(X[MONTANTS_DUS_PAYES]) * np.log1p(np.abs(X[MONTANTS_DUS_PAYES]))
    return X

def etape_log():
    return FunctionTransformer(log_montants)

def preprocesseur_lineaire():
    # Codifications : une colonne par valeur ; nombres : mise à l'échelle standard
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
        ("nombres", StandardScaler(), NUMERIQUES),
    ])

def preprocesseur_arbres():
    # Catégories : une colonne par valeur ; nombres tels quels (les arbres coupent à des seuils)
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
    ], remainder="passthrough")

MODELES = {
    "Régression logistique": make_pipeline(etape_log(), preprocesseur_lineaire(), LogisticRegression(max_iter=3000)),
    "MLP": make_pipeline(etape_log(), preprocesseur_lineaire(), MLPClassifier(max_iter=500, random_state=42)),
    "Arbre de décision": make_pipeline(etape_log(), preprocesseur_arbres(), DecisionTreeClassifier(max_depth=5, random_state=42)),
    "RandomForest": make_pipeline(etape_log(), preprocesseur_arbres(), RandomForestClassifier(random_state=42, n_jobs=-1)),
    "CatBoost": make_pipeline(etape_log(), CatBoostClassifier(loss_function="MultiClass", cat_features=CATEGORIELLES, random_seed=42, verbose=0, thread_count=-1)),
}

# Train / validation : 80/20 à l'intérieur du train (le test reste de côté)
X_app, X_val, y_app, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
print(f"Train : {len(X_app)} clients ; validation : {len(X_val)} clients ; test (de côté) : {len(X_test)} clients")

def ligne(nom, predit_app, predit_val, duree="—"):
    app, val = bonnes_predictions(y_app, predit_app), bonnes_predictions(y_val, predit_val)
    resultat = {"Modèle": nom,
                "Train : bonnes prédictions (%)": round(app["global"], 1),
                "Validation : bonnes prédictions (%)": round(val["global"], 1)}
    resultat.update({f"Validation, famille {NOMS_FAMILLES[f]} (%)": round(val[f], 1) for f in FAMILLES})
    resultat["Durée d'apprentissage (s)"] = duree
    return resultat

lignes = []
for nom, modele in MODELES.items():
    debut = time.perf_counter()
    modele.fit(X_app, y_app)
    duree = round(time.perf_counter() - debut, 1)
    lignes.append(ligne(nom, modele.predict(X_app), modele.predict(X_val), duree))
    print(f"{nom} : fait")

lignes.append(ligne("Règle « même codification qu'en août »", X_app["PAY_2"], X_val["PAY_2"]))
lignes.append(ligne("Toujours 0", np.zeros(len(X_app)), np.zeros(len(X_val))))
display(pd.DataFrame(lignes).set_index("Modèle"))

Train : 16839 clients ; validation : 4210 clients ; test (de côté) : 5263 clients
Régression logistique : fait
MLP : fait
Arbre de décision : fait
RandomForest : fait
CatBoost : fait


,Train : bonnes prédictions (%),Validation : bonnes prédictions (%),"Validation, famille -2 (%)","Validation, famille -1 (%)","Validation, famille 0 (%)","Validation, famille 2 et plus (%)",Durée d'apprentissage (s)
Modèle,,,,,,,
Régression logistique,90.8,91.0,93.2,87.7,97.5,64.9,0.6
MLP,93.1,91.0,94.6,87.5,97.0,65.7,17.0
Arbre de décision,90.5,90.9,94.1,90.9,96.9,59.9,0.1
RandomForest,100.0,91.7,94.3,90.8,97.4,64.3,0.2
CatBoost,93.4,91.7,94.1,89.9,97.6,64.7,62.9
Règle « même codification qu'en août »,89.7,89.9,91.6,83.1,97.1,66.5,—
Toujours 0,56.0,56.0,0.0,0.0,100.0,0.0,—


**Comment lire** : une ligne par modèle. « Bonnes prédictions » = part des clients rangés dans la bonne famille (-2, -1, 0 ou 2 et plus) ; les quatre colonnes « famille » donnent cette part parmi les clients de chaque famille, en validation.

**Comment évaluer**, dans cet ordre :
1. **La colonne validation** : c'est la note qui compte, sur des clients que le modèle n'a pas vus. Elle doit dépasser nettement « toujours 0 » (le plancher) et « même codification qu'en août » (la règle naïve).
2. **L'écart entre train et validation** : si le modèle fait beaucoup mieux sur le train que sur la validation, il a appris ses clients par cœur (surapprentissage) ; sa note de validation est la seule fiable.
3. **Les quatre familles** : un bon pourcentage global peut cacher une famille mal reconnue, puisque 0 est majoritaire. Le modèle doit reconnaître les quatre, en particulier les familles où la règle naïve échoue : les clients qui changent de codification entre août et septembre.

**Les clients qui changent de famille entre août et septembre (validation)**

Neuf clients sur dix gardent la même famille d'août à septembre : la règle « même codification qu'en août » est forte par construction, et le pourcentage global ne dit pas ce qu'un modèle apporte. Ici, seuls les clients de la validation **qui changent de famille** sont gardés ; la règle naïve n'en retrouve aucun, par construction. Trois types de changement :
- **sortie du retard** : 2 et plus en août, sain (-2, -1 ou 0) en septembre ; c'est la question posée par les codifications 1 qui suivent un retard (sortie ou retard maintenu ?) ;
- **nouveau retard** : sain en août, 2 et plus en septembre ;
- **changement entre codifications saines** : par exemple 0 en août, -1 en septembre.

Une ligne de plus, qui n'est pas un changement : les **retards maintenus** (2 et plus en août et en septembre). La règle naïve les retrouve tous ; un modèle qui trouve des sorties du retard peut en classer une partie à tort en sortie, et cette ligne mesure ce que cela coûte.

In [6]:
famille_aout, famille_sept = famille_pay(X_val["PAY_2"]), famille_pay(y_val)
change = famille_aout != famille_sept
TYPES_CHANGEMENT = {
    "Sortie du retard (2 et plus en août, sain en septembre)": (famille_aout == 2) & (famille_sept != 2),
    "Retard maintenu (2 et plus en août et en septembre ; pas un changement)": (famille_aout == 2) & (famille_sept == 2),
    "Nouveau retard (sain en août, 2 et plus en septembre)": (famille_aout != 2) & (famille_sept == 2),
    "Changement entre codifications saines (par exemple 0 → -1)": change & (famille_aout != 2) & (famille_sept != 2),
    "Tous les clients qui changent": change,
}
predictions_val = {nom: np.asarray(modele.predict(X_val)).ravel().astype(int) for nom, modele in MODELES.items()}
predictions_val["Règle « même codification qu'en août »"] = X_val["PAY_2"].to_numpy()
familles_predites = {nom: famille_pay(p) for nom, p in predictions_val.items()}

lignes = []
for type_changement, masque in TYPES_CHANGEMENT.items():
    resultat = {"Type de changement": type_changement, "Clients": int(masque.sum())}
    for nom, predites in familles_predites.items():
        resultat[f"{nom} : retrouvés (%)"] = round((predites[masque] == famille_sept[masque]).mean() * 100, 1)
    lignes.append(resultat)
display(pd.DataFrame(lignes).set_index("Type de changement"))

,Clients,Régression logistique : retrouvés (%),MLP : retrouvés (%),Arbre de décision : retrouvés (%),RandomForest : retrouvés (%),CatBoost : retrouvés (%),Règle « même codification qu'en août » : retrouvés (%)
Type de changement,,,,,,,
"Sortie du retard (2 et plus en août, sain en septembre)",78,67.9,76.9,83.3,80.8,80.8,0.0
Retard maintenu (2 et plus en août et en septembre ; pas un changement),333,97.6,97.0,90.1,96.7,97.3,100.0
"Nouveau retard (sain en août, 2 et plus en septembre)",168,0.0,3.6,0.0,0.0,0.0,0.0
Changement entre codifications saines (par exemple 0 → -1),181,9.9,24.3,9.9,18.2,22.1,0.0
Tous les clients qui changent,427,16.6,25.8,19.4,22.5,24.1,0.0


**Comment lire** : une ligne par type de changement, avec le nombre de clients de la validation concernés ; pour chaque modèle, la part de ces clients rangés dans leur vraie famille de septembre. La règle naïve est à 0 % sur les changements et à 100 % sur les retards maintenus. Un modèle qui en retrouve une part réelle apporte quelque chose ; s'il n'en retrouve presque aucun, régler ses paramètres ne servira pas. Les lignes à peu de clients (quelques dizaines) ne donnent qu'une tendance : un client y pèse plusieurs points.

**Les niveaux de retard attribués (validation)** : les modèles apprennent les codifications détaillées. Ce tableau dit, pour chaque niveau de retard (2 à 8), combien de clients de la validation l'ont vraiment, combien le reçoivent de chaque modèle, et combien de ces attributions sont justes. Un modèle qui n'attribue jamais 3 ou plus ramène tous les retards à 2.

In [7]:
NIVEAUX = [c for c in CLASSES if c >= 2]
vrai_val = y_val.to_numpy()
lignes = []
for niveau in NIVEAUX:
    resultat = {"Codification": niveau, "Vraies (validation)": int((vrai_val == niveau).sum())}
    for nom, p in predictions_val.items():
        resultat[f"{nom} : attribuées"] = int((p == niveau).sum())
        resultat[f"{nom} : dont justes"] = int(((p == niveau) & (vrai_val == niveau)).sum())
    lignes.append(resultat)
display(pd.DataFrame(lignes).set_index("Codification"))

,Vraies (validation),Régression logistique : attribuées,Régression logistique : dont justes,MLP : attribuées,MLP : dont justes,Arbre de décision : attribuées,Arbre de décision : dont justes,RandomForest : attribuées,RandomForest : dont justes,CatBoost : attribuées,CatBoost : dont justes,Règle « même codification qu'en août » : attribuées,Règle « même codification qu'en août » : dont justes
Codification,,,,,,,,,,,,,
2,427,314,245,313,237,314,241,297,239,301,245,359,246
3,52,13,6,36,11,0,0,26,10,24,8,37,9
4,12,11,4,10,2,0,0,6,1,5,1,9,1
5,4,7,4,2,0,0,0,3,1,3,1,2,0
6,2,1,1,0,0,0,0,2,2,2,2,1,0
7,1,1,1,1,1,0,0,1,1,1,1,3,0
8,3,3,3,3,3,0,0,3,3,3,3,0,0


**Comment lire** : une ligne par niveau de retard. « Vraies » : clients de la validation qui ont ce niveau en septembre ; « attribuées » : clients à qui le modèle donne ce niveau ; « dont justes » : parmi eux, ceux qui l'ont vraiment. La règle « même codification qu'en août » sert de repère : elle attribue le niveau d'août.